# Phase 2: Baseline Models

## Notebook Sections
- Train/validation split setup
- Baseline preprocessing
- Baseline model runs
- CV/validation comparison
- Key baseline takeaways

In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import ElasticNet
from sklearn.metrics import r2_score


In [3]:
df = pd.read_csv("../data/CW1_train.csv")

TARGET = "outcome"
X = df.drop(columns=[TARGET])
y = df[TARGET]


In [4]:
cat_cols = X.select_dtypes(include="object").columns.tolist()
num_cols = X.select_dtypes(exclude="object").columns.tolist()

print("Categorical:", cat_cols)
print("Numerical:", len(num_cols))


Categorical: ['cut', 'color', 'clarity']
Numerical: 27


/var/folders/ps/8pn5r2xx4nnggpd_z9t92cww0000gn/T/ipykernel_34944/2837319812.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X.select_dtypes(include="object").columns.tolist()


In [5]:
class MeanTargetEncoder:
    def __init__(self, cols):
        self.cols = cols
        self.global_mean = None
        self.maps = {}

    def fit(self, X, y):
        self.global_mean = y.mean()
        for col in self.cols:
            stats = pd.concat([X[col], y], axis=1).groupby(col)[y.name].mean()
            self.maps[col] = stats
        return self

    def transform(self, X):
        X_new = X.copy()
        for col in self.cols:
            X_new[col] = X_new[col].map(self.maps[col]).fillna(self.global_mean)
        return X_new


In [6]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [7]:
num_pipeline = Pipeline(
    [
        ("scaler", StandardScaler()),
    ]
)


In [8]:
encoder = MeanTargetEncoder(cat_cols)

X_train_enc = encoder.fit(X_train, y_train).transform(X_train)
X_val_enc = encoder.transform(X_val)

X_train_enc[num_cols] = num_pipeline.fit_transform(X_train_enc[num_cols])
X_val_enc[num_cols] = num_pipeline.transform(X_val_enc[num_cols])


In [9]:
model = ElasticNet(alpha=0.05, l1_ratio=0.5, random_state=42)

model.fit(X_train_enc, y_train)

preds = model.predict(X_val_enc)
r2 = r2_score(y_val, preds)

print("Validation R2:", round(r2, 4))


Validation R2: 0.271


In [10]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

cv_scores = cross_val_score(
    model,
    X_train_enc,
    y_train,
    cv=kf,
    scoring="r2"
)

print("CV R2 mean:", cv_scores.mean())


CV R2 mean: 0.2889951297815606


In [11]:
print("CV R2 mean:", cv_scores.mean())
print("CV R2 std:", cv_scores.std())

CV R2 mean: 0.2889951297815606
CV R2 std: 0.018354852129871466
